# 03 · Phishing URL Detection

Build and compare ML classifiers to detect phishing websites using URL-based features.

**Pipeline:**
1. Extract 28 structural features from raw URLs (length, special chars, domain patterns, etc.)
2. Build a labelled dataset of phishing + legitimate URLs
3. Train 7+ classifiers and compare performance
4. Classify any URL via the trained model

In [ ]:
import sys, os, math, json, re, warnings
from pathlib import Path
from collections import Counter
from urllib.parse import urlparse, parse_qs
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

warnings.filterwarnings('ignore')
sns.set_style('darkgrid')
plt.rcParams.update({'figure.facecolor': '#1a1a2e', 'axes.facecolor': '#1a1a2e',
                     'axes.edgecolor': '#4a4a6a', 'text.color': '#e0e0e0',
                     'axes.labelcolor': '#e0e0e0', 'xtick.color': '#a0a0b0',
                     'ytick.color': '#a0a0b0'})

print('Environment ready.')

---
## 1. URL Feature Extraction

We extract 28 features from each URL that help discriminate phishing sites from legitimate ones.

In [ ]:
from url_feature_extractor import extract_features, FEATURE_NAMES, dataframe_from_urls

# Demo: extract features from a few URLs
demo_urls = [
    'https://www.google.com/search?q=hello',
    'http://192.168.1.1/login.php?cmd=verify',
    'http://bit.ly/3abcde',
    'https://paypal-secure-login.com/update/account/verify',
    'https://github.com/features/actions',
]

demo_df = dataframe_from_urls(demo_urls)
demo_df.insert(0, 'url', demo_urls)
demo_df['label'] = ['legitimate', 'phishing', 'phishing', 'phishing', 'legitimate']
demo_df

In [ ]:
# Show the feature vector for a single URL
url = 'https://paypal-secure-login.com/update/account/verify'
feats = extract_features(url)
print(f'URL: {url}\n')
print(f'{"Feature":30s} {"Value":>10s}')
print('-' * 42)
for k, v in sorted(feats.items(), key=lambda x: -abs(x[1])):
    if abs(v) > 0 or k in ('has_https', 'has_www'):
        print(f'{k:30s} {v:>10}')

---
## 2. Build the Dataset

Generate phishing URLs from open-source feeds and legitimate URLs from a curated top-domain list.

In [ ]:
# Build the dataset (downloads/generates URLs, extracts features)
from dataset_builder import main as build_dataset
build_dataset()

# Load the saved feature matrix
df = pd.read_csv('data/phishing_features.csv')
print(f'Dataset: {df.shape[0]} samples, {df.shape[1] - 1} features')
print(f'Classes:\n  Legitimate (0): {(df["label"] == 0).sum()}\n  Phishing (1):   {(df["label"] == 1).sum()}')

In [ ]:
# Feature distribution comparison
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
features_to_plot = ['url_length', 'num_digits', 'has_suspicious_words', 
                    'has_ip_address', 'url_entropy', 'special_char_ratio']

for ax, feat in zip(axes.flat, features_to_plot):
    legit = df[df['label'] == 0][feat]
    phish = df[df['label'] == 1][feat]
    ax.hist(legit, bins=30, alpha=0.6, label='Legitimate', color='#22c55e', density=True)
    ax.hist(phish, bins=30, alpha=0.6, label='Phishing', color='#ef4444', density=True)
    ax.set_title(feat.replace('_', ' ').title(), fontsize=11, color='#e0e0e0')
    ax.legend(fontsize=8, loc='upper right')

fig.suptitle('Feature Distributions: Legitimate vs Phishing', fontsize=14, color='#e0e0e0', y=1.02)
fig.tight_layout()
plt.show()

---
## 3. Train & Compare Models

Train multiple classifiers and compare their performance on held-out test data.

In [ ]:
%%time
from model_comparison import main as run_models
run_models()

In [ ]:
# Load results and visualize
with open('results/metrics.json') as f:
    metrics = json.load(f)

results_df = pd.DataFrame(metrics).T.reset_index().rename(columns={'index': 'Model'})

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
for ax, metric in zip(axes, ['f1', 'roc_auc', 'accuracy']):
    sorted_df = results_df.sort_values(metric, ascending=True)
    colors = ['#00d4aa' if v == sorted_df[metric].max() else '#4a4a6a' for v in sorted_df[metric]]
    ax.barh(sorted_df['Model'], sorted_df[metric], color=colors, edgecolor='#2a2a4a', linewidth=0.5)
    ax.set_title(metric.upper(), fontsize=12, color='#e0e0e0')
    ax.set_xlabel('Score', color='#a0a0b0')
    for i, v in enumerate(sorted_df[metric]):
        ax.text(v + 0.003, i, f'{v:.4f}', va='center', fontsize=9, color='#a0a0b0')

fig.suptitle('Model Comparison — Phishing URL Detection', fontsize=14, color='#e0e0e0', y=1.05)
fig.tight_layout()
plt.show()

In [ ]:
# Show detailed comparison table
results_df[['Model', 'accuracy', 'precision', 'recall', 'f1', 'roc_auc', 'train_time_s']].sort_values('f1', ascending=False)

---
## 4. Classify URLs

Use the trained model to classify new URLs.

In [ ]:
from phishing_classifier import classify_url

test_urls = [
    'https://www.github.com',
    'http://secure-paypal-login.xyz/webscr?cmd=login',
    'https://stackoverflow.com/questions/ask',
    'http://bit.ly/3abcde',
    'http://192.168.1.1/login.php',
    'https://www.google.com/search?q=phishing+detection',
]

for url in test_urls:
    classify_url(url, verbose=True)
    print()

---
## 5. Feature Importance Analysis

Which features matter most for detecting phishing URLs?

In [ ]:
from joblib import load
from url_feature_extractor import FEATURE_NAMES

pipeline = load('results/models/RandomForest.joblib')
rf = pipeline.named_steps['clf']

importances = sorted(zip(FEATURE_NAMES, rf.feature_importances_), key=lambda x: x[1], reverse=True)

fig, ax = plt.subplots(figsize=(10, 6))
names, vals = zip(*importances[:15])
ax.barh(range(len(names)), vals, color='#7c3aed', edgecolor='#4a4a6a', linewidth=0.5)
ax.set_yticks(range(len(names)))
ax.set_yticklabels([n.replace('_', ' ').title() for n in names])
ax.set_xlabel('Importance', color='#a0a0b0')
ax.set_title('Top-15 Feature Importances (Random Forest)', fontsize=13, color='#e0e0e0')
ax.invert_yaxis()
fig.tight_layout()
plt.show()

---
## Summary

**What we built:**
- Feature extractor: 28 URL/domain-based features from raw URLs
- Dataset builder: phishing URLs from open sources + legitimate URLs
- 7+ classifiers trained and compared
- CLI classifier for URL prediction
- HTML report with charts and performance metrics

**Key findings:**
- Ensemble methods (Voting Ensemble, Random Forest) typically perform best
- URL entropy, suspicious words, and digit ratio are strong phishing indicators
- Simple models like Logistic Regression provide competitive baselines